# Deepfake detector — Colab pipeline (calibration refit + AASIST-L fine-tune)

**Runtime → Change runtime type → T4 GPU.**

Before you start, put these in Google Drive at `MyDrive/deepfake_detector/`:

| File | How to make it | Needed for |
|---|---|---|
| `colab_data.zip` | on the laptop: `.venv\Scripts\python.exe scripts/pack_for_colab.py` (~3.5 GB) | everything |
| `SSL_AASIST_DF.pth` (optional) | copy from `outputs/weights/` (1.2 GB) | step 6 only |

Results (calibrators, fine-tuned weights, score CSVs) are copied back to the same Drive folder.
Score CSVs are written straight to Drive, so if Colab disconnects, re-run the cells: they resume.

## 1. GPU + Drive

In [ ]:
!nvidia-smi -L
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/deepfake_detector'
import os; os.makedirs(f'{DRIVE}/scores', exist_ok=True)
!ls -lh "{DRIVE}"

## 2. Code (branch `sg`)
If the repo is private: `git clone https://<user>:<token>@github.com/...` with a GitHub personal access token.

In [ ]:
%cd /content
!git clone -b sg https://github.com/abhishek1543748/MinorProject.git || (cd MinorProject && git pull)
%cd /content/MinorProject
!git log --oneline -3

## 3. Dependencies
Colab already has CUDA torch/torchaudio — do not reinstall them. `av` is the decode fallback if torchaudio's backend can't read a file.

In [ ]:
!pip -q install av "transformers>=4.40" soundfile scikit-learn fastapi python-multipart
import torch, torchaudio; print(torch.__version__, torchaudio.__version__, torch.cuda.is_available())

## 4. Data
Copy the zip to local disk first (unzipping straight from Drive is very slow).

In [ ]:
!mkdir -p data/raw
!cp "{DRIVE}/colab_data.zip" /content/colab_data.zip
!unzip -q -n /content/colab_data.zip -d data/raw && rm /content/colab_data.zip
!ls data/raw data/raw/ASVspoof2019_LA
!ls data/raw/ASVspoof2019_LA/ASVspoof2019_LA_dev/flac | wc -l

In [ ]:
# sanity: preprocessing parity (GOLDEN RULE) + refit/quantize unit tests
!python -m pytest tests/test_preprocess.py tests/test_refit_calibrator.py -q -p no:warnings

## 5. Refit AASIST-L calibrator on window-mean scores
Fixes the mismatch: `calibrator_mixed.pkl` was fitted on single 4 s windows but the server feeds it the mean over windows.
Sets = LA dev + DF-cal speakers (DF-test speakers stay held out). ~1 h on Colab (audio decoding is the bottleneck).
Writes a NEW file name so nothing is overwritten.

In [ ]:
!python scripts/refit_windowed_calibrator.py --device cuda \
    --scores-csv "{DRIVE}/scores/windowed_calib_aasist_l_scores.csv" \
    --out outputs/weights/calibrator_mixed_windowed.pkl
!cp outputs/weights/calibrator_mixed_windowed.pkl "{DRIVE}/"

## 6. (Optional) SSL-AASIST calibrator
INT8 quantization is CPU-only, so on GPU this fits on fp32 scores. INT8 scores differ by < 0.35 LLR, and the
server falls back to `calibrator_ssl.pkl` when `calibrator_ssl_int8.pkl` doesn't exist — fine for the demo.
`--max_files 4000` per set keeps it ~30-40 min; drop it for the full set.

In [ ]:
!cp "{DRIVE}/SSL_AASIST_DF.pth" outputs/weights/
!python scripts/refit_windowed_calibrator.py --model ssl --device cuda --max_files 4000 \
    --scores-csv "{DRIVE}/scores/windowed_calib_ssl_scores.csv" \
    --out outputs/weights/calibrator_ssl.pkl
!cp outputs/weights/calibrator_ssl.pkl "{DRIVE}/"

## 7. Fetch ~2,500 diverse real speech clips (LibriSpeech dev-other, streamed, ~300 MB)

In [ ]:
!python scripts/fetch_bonafide_diversity.py --n 2500
!wc -l data/raw/bonafide_diversity/protocol_*.txt

## 8. Fine-tune AASIST-L (6 epochs, CE, lr 1e-5, RawBoost type 5) — ~1 h
- `--no-cache`: with DataLoader workers the dev cache is per-worker and just doubles RAM.
- A checkpoint is written only if an epoch beats the starting model's dev EER.

In [ ]:
# DF-cal / DF-test protocols (speaker-disjoint, same split as Phase 5b)
!python scripts/make_df_protocols.py
# Best checkpoint is chosen by EER on DF-cal (LA dev EER still logged).
# The last epoch is ALWAYS saved as finetuned_aasist_l_last.pth.
# Baselines for stock AASIST-L: LA dev 1.31%, DF-cal 14.89%, DF-test 16.37%.
!python train.py --config configs/finetune_config.json \
    --resume_weights outputs/weights/AASIST-L.pth \
    --extra-train-protocol data/raw/bonafide_diversity/protocol_train.txt \
    --extra-train-audio-dir data/raw/bonafide_diversity/flac \
    --extra-dev-protocol data/raw/bonafide_diversity/protocol_dev.txt \
    --extra-dev-audio-dir data/raw/bonafide_diversity/flac \
    --select-protocol data/raw/ASVspoof2021_DF/df_cal_protocol.txt \
    --select-audio-dir data/raw/ASVspoof2021_DF/ASVspoof2021_DF_eval/flac \
    --workers 2 --no-cache --epochs 3 --save-name finetuned_aasist_l.pth
!cp outputs/weights/finetuned_aasist_l*.pth outputs/scores/finetuned_aasist_l_log.csv "{DRIVE}/"


## 9. Did it help? Cross-domain EER on the 2021 DF sample (baseline AASIST-L: 15.74%)

In [ ]:
# Held-out report on DF-test (speakers never used for selection or calibration).
# Stock AASIST-L on the same files: EER 16.37%.
!python eval.py --device cuda \
    --protocol data/raw/ASVspoof2021_DF/df_test_protocol.txt \
    --audio-dir data/raw/ASVspoof2021_DF/ASVspoof2021_DF_eval/flac \
    --weights outputs/weights/finetuned_aasist_l.pth --run-name ft_df_test
!cat outputs/scores/ft_df_test_summary.txt
!cp outputs/scores/ft_df_test_* "{DRIVE}/scores/"


## 10. Calibrator for the fine-tuned weights (scores changed, so the old calibrator no longer fits)

In [ ]:
!python scripts/refit_windowed_calibrator.py --device cuda \
    --weights outputs/weights/finetuned_aasist_l.pth \
    --scores-csv "{DRIVE}/scores/windowed_calib_finetuned_scores.csv" \
    --out outputs/weights/calibrator_finetuned.pkl
!cp outputs/weights/calibrator_finetuned.pkl "{DRIVE}/"
!ls -lh "{DRIVE}"

## Back on the laptop
Download from Drive into `outputs/weights/`, then:

```powershell
# window-mean calibrator for the stock AASIST-L
$env:DFD_CALIBRATOR="outputs/weights/calibrator_mixed_windowed.pkl"
.venv\Scripts\python.exe -m uvicorn src.api.server:app --port 8000

# fine-tuned model: server loads best_model.pth if present
copy outputs\weights\finetuned_aasist_l.pth outputs\weights\best_model.pth
$env:DFD_CALIBRATOR="outputs/weights/calibrator_finetuned.pkl"

# SSL INT8 demo (uses calibrator_ssl.pkl)
$env:DFD_MODEL="ssl"; $env:DFD_QUANTIZE="1"
```
Only keep the fine-tuned model if its DF-test EER is clearly below 16.37% (stock AASIST-L on the same files) and LA dev EER stayed acceptable (stock: 1.31%).